In [0]:
%python
from pyspark.sql.functions import col, when, lower
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

data_path = "/Volumes/barbara_lakehouse/ml_sandbox/data/train.csv"
df = spark.read.csv(data_path, header=True, inferSchema=True)

def bool_to_int(c):
    return when(lower(col(c)) == "true", 1)\
           .when(lower(col(c)) == "false", 0)\
           .otherwise(None)

df = df.withColumn("PassengerId", col("PassengerId").cast("string")) \
       .withColumn("VIP", bool_to_int("VIP")) \
       .withColumn("CryoSleep", bool_to_int("CryoSleep")) \
       .withColumn("label", bool_to_int("Transported").cast("double"))

numerical_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
categorical_cols = ["HomePlanet", "Destination", "VIP", "CryoSleep"]

imputer = Imputer(
    inputCols=numerical_cols,
    outputCols=[c + "_imputed" for c in numerical_cols]
)

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in categorical_cols],
    outputCols=[c + "_ohe" for c in categorical_cols],
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=[c + "_imputed" for c in numerical_cols] + [c + "_ohe" for c in categorical_cols],
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=300,
    maxDepth=8,
    featureSubsetStrategy="sqrt",
    subsamplingRate=0.8,
    seed=42
)

pipeline = Pipeline(stages=[imputer] + indexers + [encoder, assembler, rf])

train, valid = df.randomSplit([0.8, 0.2], seed=42)
rf_model = pipeline.fit(train)

pred = rf_model.transform(valid)
evaluator = BinaryClassificationEvaluator(labelCol="label")
print("Validation AUC:", evaluator.evaluate(pred))

pred.select("PassengerId", "label", "probability", "prediction").show(10, truncate=False)

In [0]:
%python
test_path = "/Volumes/barbara_lakehouse/ml_sandbox/data/test.csv"
test_df = spark.read.csv(test_path, header=True, inferSchema=True)

test_df = test_df.withColumn("PassengerId", col("PassengerId").cast("string")) \
                 .withColumn("VIP", bool_to_int("VIP")) \
                 .withColumn("CryoSleep", bool_to_int("CryoSleep"))

test_pred = rf_model.transform(test_df)

submission = test_pred.select(
    col("PassengerId"),
    (col("prediction") == 1).alias("Transported")
)

out_dir = "/Volumes/barbara_lakehouse/ml_sandbox/data/spark_rf_submission"
submission.coalesce(1).write.mode("overwrite").option("header", True).csv(out_dir)

display(submission.limit(10))
print("Saved to:", out_dir)


In [0]:
%python
import os
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/barbara_lakehouse/ml_sandbox/data/sparkml_temp'

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [100])
    .addGrid(rf.maxDepth, [6])
    .build()
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(labelCol="label"),
    numFolds=2,
    parallelism=2
)

cv_model = cv.fit(train)
pred_cv = cv_model.transform(valid)
print("Best CV AUC:", BinaryClassificationEvaluator(labelCol="label").evaluate(pred_cv))